# Error Handling: Conventions and Pitfalls

Besides the general principles that we just discussed, we also want to cover a few things that can easily lure in new developers that have discovered the joys of error handling.

## Exceptions are not control flow

"Control flow" references how your program navigates your code.
Statements like `if`, `for`, `while`, `match`, etc are used to control the code-paths that your program decides to follow, based on the values being used at runtime.
Such statements exist to enable you to conditionally perform certain tasks, based on the inputs you are given.

Some of you may be aware of the existence of [the `try`/`except` block](https://www.w3schools.com/python/python_try_except.asp), which in my opinion is a trap.

![`try`/`except` is a trap and you should use it with care!](https://media2.giphy.com/media/v1.Y2lkPTc5MGI3NjExdnR5a3Q1d3RiaW54Ymt3N3YzdTB4YXJ5NnJzODZ1YTBpa3dsZ2d6ciZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/9PAIhJvcQ35hdZPUir/giphy.gif)

Exceptions - and in particular the `try`/`except` block - should not be used for control flow.
It is much better (and performant) to "check if what you are doing is right, and throw an error if it is wrong", as opposed to "try, beg forgiveness, then try something else".
For example, let's say we want to adapt our [`tri_solver` function](./principles.ipynb) from the last section so that it can accept a scalar for the vector $b$.
We could simply write our method first assuming that ("trying") $b$ is a numpy array, and if we encounter a problem try again by assuming it's a scalar.

In [ ]:
import numpy as np
from numpy.linalg import solve

In [ ]:
def tri_solver(A: np.ndarray, b: np.ndarray | float) -> np.ndarray:
    """Solve the equation

    $$ A x = b $$

    for $x$, where $A$ is an upper-triangular matrix and $b$ is a column vector
    or scalar (that will be treated as a column vector of appropriate size).

    Raises
    ------
    RuntimeError:
        If $A$ is incompatible with the solver method.
    RuntimeError:
        If $b$ is not provided as a column vector.
    """
    if np.any(np.triu(A) != A):
        raise RuntimeError("Input matrix $A$ is not upper-triangular")
    
    diag_A = np.diag(A)
    upper_tri_A = np.triu(A, k=1)
    if np.all(diag_A == sorted(diag_A, reverse=True)) and diag_A.sum() < upper_tri_A.sum():
        raise RuntimeError(
            "$A$ cannot have a decreasing diagonal that sums "
            "to less than the sum of the non-diagonal elements."
        )

    try:
        if b.size != b.shape[0]:
            raise RuntimeError(
                f"RHS vector $b$ must be a column vector (got shape {b.shape})"
            )
    except AttributeError as error:
        # float variables don't have a .size or .shape attribute,
        # so if we encounter an AttributeError we have a float for b and not
        # an array. As such, we can now cast it into an array of the correct size.
        b = b * np.ones((A.shape[0],))

    return solve(A, b)

In [ ]:
print("Pass scalar:", tri_solver(np.eye(3), 1.0))
print("Pass vector:", tri_solver(np.eye(3), np.ones((3,))))

This _works_, but it is ugly, offers poor performance, and isn't very readable however.

- We _know_ a priori what types to expect $b$ as.
  Rather than trying one case, and if we hit a problem then trying the other, we could simply create conditional paths with the two sets of instructions!
  Reading something like this is much easier than following `except` statements around.
- The code that we are wrapping in the `try`/`except` block is also explicitly `raise`-ing an error already.
  This forces us to specify what type of error we want to `except` to avoid catching our own `RuntimeError`.
- Code inside the `try` block will execute and update the program state before entering the `except` block.
  This means that the `except` block is not a "fresh restart" from the beginning of the `try` block - things might have changed unexpectedly and you will need to account for this.
  A related consideration is that you don't know _where_ in the `try` block the error was encountered - it might not be in the place you are expecting it to be!

It is much better to anticipate the problem / case, _check for it_ and handle it appropriately before continuing.

In [ ]:
def tri_solver(A: np.ndarray, b: np.ndarray | float) -> np.ndarray:
    """Solve the equation

    $$ A x = b $$

    for $x$, where $A$ is an upper-triangular matrix and $b$ is a column vector
    or scalar (that will be treated as a column vector of appropriate size).

    Raises
    ------
    RuntimeError:
        If $A$ is incompatible with the solver method.
    RuntimeError:
        If $b$ is not provided as a column vector.
    """
    if np.any(np.triu(A) != A):
        raise RuntimeError("Input matrix $A$ is not upper-triangular")

    diag_A = np.diag(A)
    upper_tri_A = np.triu(A, k=1)
    if (
        np.all(diag_A == sorted(diag_A, reverse=True))
        and diag_A.sum() < upper_tri_A.sum()
    ):
        raise RuntimeError(
            "$A$ cannot have a decreasing diagonal that sums "
            "to less than the sum of the non-diagonal elements."
        )

    # If b is not an array, we need to convert the scalar into a column vector
    # of appropriate size. Check for this case and handle it appropriately.
    if not isinstance(b, np.ndarray):
        b = b * np.ones((A.shape[0],))

    # From this point on in the code, we now know that b is a column vector.
    # Thus, we can write code as such.
    if b.size != b.shape[0]:
        raise RuntimeError(
            f"RHS vector $b$ must be a column vector (got shape {b.shape})"
        )

    return solve(A, b)

print("Pass scalar:", tri_solver(np.eye(3), 1.0))
print("Pass vector:", tri_solver(np.eye(3), np.ones((3,))))

Same result, but it is easier to understand the flow control in this implementation.

## Fail-Fast

Raising an exception terminates the program.
This means that any time (or resources) the program has consumed / spent thus far are going to be wasted / lost, since it was unable to finish it's task.
I'm fairly sure everyone has - at some point in their career - left a job running over the weekend, only to come back in on Monday to find that the simulation ran perfectly... but the output could not be saved!

**Note**: "resources" here specifically refers to runtime resources.
If you were writing out to intermediate files as you went along for example, they'll still persist (assuming they were specified correctly!).
But you might have been hogging several GPUs and a large amount of RAM whilst your program was running, which will make you very unpopular at the next group meeting if you turn up with nothing to show for it.

Given that this is going to happen, you should aim to have your program "fail-fast" where possible.
This means that, if you are considering raising an exception, you should do it at the earliest possible opportunity, to avoid wasting resources that could otherwise be spent somewhere else.

A typical example that is often seen is providing an invalid path to save an output to, but only checking the validity of this path once the program has actually performed its computations.
Taking our favourite `tri_solver` function as an example;

In [ ]:
from pathlib import Path

In [ ]:
def tri_solver(A: np.ndarray, b: np.ndarray | float, save_to: Path | None = None) -> np.ndarray:
    """Solve the equation

    $$ A x = b $$

    for $x$, where $A$ is an upper-triangular matrix and $b$ is a column vector
    or scalar (that will be treated as a column vector of appropriate size).

    Save the resulting vector to the directory `save_to`, if provided.

    Raises
    ------
    RuntimeError:
        If $A$ is incompatible with the solver method.
    RuntimeError:
        If $b$ is not provided as a column vector.
    """
    if np.any(np.triu(A) != A):
        raise RuntimeError("Input matrix $A$ is not upper-triangular")

    diag_A = np.diag(A)
    upper_tri_A = np.triu(A, k=1)
    if (
        np.all(diag_A == sorted(diag_A, reverse=True))
        and diag_A.sum() < upper_tri_A.sum()
    ):
        raise RuntimeError(
            "$A$ cannot have a decreasing diagonal that sums "
            "to less than the sum of the non-diagonal elements."
        )

    # If b is not an array, we need to convert the scalar into a column vector
    # of appropriate size. Check for this case and handle it appropriately.
    if not isinstance(b, np.ndarray):
        b = b * np.ones((A.shape[0],))

    # From this point on in the code, we now know that b is a column vector.
    # Thus, we can write code as such.
    if b.size != b.shape[0]:
        raise RuntimeError(
            f"RHS vector $b$ must be a column vector (got shape {b.shape})"
        )

    # Wrapping the 'solve' in print statements to illustrate execution
    print("Beginning computation")
    x = solve(A, b)
    print("Computation done")

    if save_to is not None:
        # Note that np.save will actually perform this check for us, so we are
        # violating the principle of "delegate errors when possible" here, just
        # for illustration.
        if save_to.exists() and save_to.is_dir():
            np.save(save_to / "results.npz", x)
        else:
            raise RuntimeError(f"Directory {save_to} does not exist")
    
    return x

tri_solver(np.eye(3), 1.0, Path("/i/dont/exist/why/did/you/try/this"))

However, writing the method in this way will mean that our `solve` will always run, which will consume compute resources (notice the `print` outputs that we get before the stacktrace).
And then these will all have been wasted if afterwards, we find that the location to save to never actually existed and the error is thrown.

But we _knew_ what could go wrong right when we got given `save_to` in the first place!
So we should be failing fast: we should check whether `save_to` is OK to use as soon as possible, which means that we'll only proceed with the resource-intensive `solve` if we know we're going to be able to use the result.

In [ ]:
def tri_solver(
    A: np.ndarray, b: np.ndarray | float, save_to: Path | None = None
) -> np.ndarray:
    """Solve the equation

    $$ A x = b $$

    for $x$, where $A$ is an upper-triangular matrix and $b$ is a column vector
    or scalar (that will be treated as a column vector of appropriate size).

    Save the resulting vector to the directory `save_to`, if provided.

    Raises
    ------
    RuntimeError:
        If $A$ is incompatible with the solver method.
    RuntimeError:
        If $b$ is not provided as a column vector.
    """
    if save_to is not None and not save_to.is_dir():
        raise RuntimeError(f"Directory {save_to} does not exist")

    if np.any(np.triu(A) != A):
        raise RuntimeError("Input matrix $A$ is not upper-triangular")

    diag_A = np.diag(A)
    upper_tri_A = np.triu(A, k=1)
    if (
        np.all(diag_A == sorted(diag_A, reverse=True))
        and diag_A.sum() < upper_tri_A.sum()
    ):
        raise RuntimeError(
            "$A$ cannot have a decreasing diagonal that sums "
            "to less than the sum of the non-diagonal elements."
        )

    # If b is not an array, we need to convert the scalar into a column vector
    # of appropriate size. Check for this case and handle it appropriately.
    if not isinstance(b, np.ndarray):
        b = b * np.ones((A.shape[0],))

    # From this point on in the code, we now know that b is a column vector.
    # Thus, we can write code as such.
    if b.size != b.shape[0]:
        raise RuntimeError(
            f"RHS vector $b$ must be a column vector (got shape {b.shape})"
        )

    # Wrapping the 'solve' in print statements to illustrate execution
    print("Beginning computation")
    x = solve(A, b)
    print("Computation done")
    
    if save_to is not None:
        np.save(save_to / "results.npz", x)

    return x

tri_solver(np.eye(3), 1.0, Path("/i/dont/exist/why/did/you/try/this"))

Notice that this time, we avoided running the potentially expensive `solve` process.

## Custom Exceptions

`Exception` is a Python class, which can be inherited from.
This means that it is possible to define your own "error types" by creating subclasses of `Exception` in your codebase.
Typically you'll want to define all your custom errors in one place (particularly if you are writing a package) - like [in a dedicated sub-module](https://github.com/NSs-FLF-RHUL/gains/blob/main/src/gains/exceptions.py).
You typically _don't_ want to be defining custom exceptions in scripts.

Continuing our `tri_solver` example above, we have been using `RuntimeError`s throughout because there isn't a built-in `Exception` type that really covers our use case.
We might choose to define our own `Exception` type to catch all these provisos in our method.

In [ ]:
class CantSolveError(Exception):
    """Exception type raised by `tri_solve` when provided inputs that cannot be
    used with the solver method.
    """

Note that it is good practice to include a docstring on your custom exception class, so users (and yourself) can remind themselves of when to use your exception type!

Once defined, we can use `CantSolveError` in our code:

In [ ]:
def tri_solver(
    A: np.ndarray, b: np.ndarray | float, save_to: Path | None = None
) -> np.ndarray:
    """Solve the equation

    $$ A x = b $$

    for $x$, where $A$ is an upper-triangular matrix and $b$ is a column vector
    or scalar (that will be treated as a column vector of appropriate size).

    Save the resulting vector to the directory `save_to`, if provided.

    Raises
    ------
    RuntimeError:
        If $A$ is incompatible with the solver method.
    RuntimeError:
        If $b$ is not provided as a column vector.
    """
    if save_to is not None and not save_to.is_dir():
        raise RuntimeError(f"Directory {save_to} does not exist")

    if np.any(np.triu(A) != A):
        raise CantSolveError("Input matrix $A$ is not upper-triangular")

    diag_A = np.diag(A)
    upper_tri_A = np.triu(A, k=1)
    if (
        np.all(diag_A == sorted(diag_A, reverse=True))
        and diag_A.sum() < upper_tri_A.sum()
    ):
        raise CantSolveError(
            "$A$ cannot have a decreasing diagonal that sums "
            "to less than the sum of the non-diagonal elements."
        )

    # If b is not an array, we need to convert the scalar into a column vector
    # of appropriate size. Check for this case and handle it appropriately.
    if not isinstance(b, np.ndarray):
        b = b * np.ones((A.shape[0],))

    # From this point on in the code, we now know that b is a column vector.
    # Thus, we can write code as such.
    if b.size != b.shape[0]:
        raise CantSolveError(
            f"RHS vector $b$ must be a column vector (got shape {b.shape})"
        )

    # Wrapping the 'solve' in print statements to illustrate execution
    print("Beginning computation")
    x = solve(A, b)
    print("Computation done")

    if save_to is not None:
        np.save(save_to / "results.npz", x)

    return x

# Give a non-upper triangular matrix to hit one of our error cases
tri_solver(np.arange(9).reshape((3,3)), 1.0)

Notice that now, we get a `CantSolveError` from the stacktrace!

### Why don't I do this all the time?

The problem with custom exception types is precisely that they are _custom_ - that is, non-standard - exception types.
A user will typically have an intuition for what each of the built-in exception types correspond to, but they won't have any intuition about what kinds of problems your custom exception types correspond to.
This doesn't mean that you should _never_ use custom exceptions, but it does mean you need to really consider how you name them and take extra care to ensure any error messages you give them are explanatory enough.

Another issue with defining custom exceptions is precisely that you need to store their definition somewhere!
As mentioned previously, in a package you'll typically do this within a single sub-module, and `import` your custom exception type when you want to raise it.
You shouldn't define custom exception types in scripts for this same reason - you shouldn't be `import`-ing objects into scripts from other scripts.

Lastly, it is very easy to fall into the trap of having a custom exception type for _every possible type of problem_ in your codebase.
This often results in a very long file of custom exceptions, and just opens up the possibility that your use of these custom exceptions won't be consistent.

If you do need to use custom exceptions, I would encourage you to err on the side of caution and "less is more".
When writing a package that needs a custom exception, a good idea is simply to create a single exception that your package will use whenever it encounters an issue that isn't covered by the built-in exception types.
For our `tri_solver` example, I would consider it better practice to have defined a custom exception like the below:

In [ ]:
class TriSolverError(Exception):
    """Exception type raised by `tri_solve` when encountering an issue at runtime
    concerning the execution or application of the solver method.
    """

def tri_solver(
    A: np.ndarray, b: np.ndarray | float, save_to: Path | None = None
) -> np.ndarray:
    """Solve the equation

    $$ A x = b $$

    for $x$, where $A$ is an upper-triangular matrix and $b$ is a column vector
    or scalar (that will be treated as a column vector of appropriate size).

    Save the resulting vector to the directory `save_to`, if provided.

    Raises
    ------
    RuntimeError:
        If $A$ is incompatible with the solver method.
    RuntimeError:
        If $b$ is not provided as a column vector.
    """
    if save_to is not None and not save_to.is_dir():
        raise FileNotFoundError(f"Directory {save_to} does not exist")

    if np.any(np.triu(A) != A):
        raise TriSolverError("Input matrix $A$ is not upper-triangular")

    diag_A = np.diag(A)
    upper_tri_A = np.triu(A, k=1)
    if (
        np.all(diag_A == sorted(diag_A, reverse=True))
        and diag_A.sum() < upper_tri_A.sum()
    ):
        raise TriSolverError(
            "$A$ cannot have a decreasing diagonal that sums "
            "to less than the sum of the non-diagonal elements."
        )

    # If b is not an array, we need to convert the scalar into a column vector
    # of appropriate size. Check for this case and handle it appropriately.
    if not isinstance(b, np.ndarray):
        b = b * np.ones((A.shape[0],))

    # From this point on in the code, we now know that b is a column vector.
    # Thus, we can write code as such.
    if b.size != b.shape[0]:
        raise TriSolverError(
            f"RHS vector $b$ must be a column vector (got shape {b.shape})"
        )

    # Wrapping the 'solve' in print statements to illustrate execution
    print("Beginning computation")
    x = solve(A, b)
    print("Computation done")

    if save_to is not None:
        np.save(save_to / "results.npz", x)

    return x

This way of writing the code means that:

- When we encounter a problem that is covered by a general Python exception, we can `raise` that exception type with appropriate help.
  This means that the user can apply any understanding they have of that particular error to resolve the problem.
  The stacktrace will indicate that the line on which the error occurred is inside our code, but our use of a build-in exception is implicitly us saying "this is a problem that arises in Python in general, it's not specifically our fault".
- When we encounter a situation that is of our own making - such as the "niche" conditions for our solver method to work - we can use our `TriSolverError`.
  Using this error type at least immediately indicates to a user that they are using our code incorrectly and should look into our documentation and error messages carefully, because this type of exception is clearly custom to our code (as inferred from its name).
  In general for your own packages I recommend using an analogous custom exception named something like `<MyPackageName>Error` if you do want to use custom exceptions.

In [ ]:
# Give a non-upper triangular matrix to hit an error case that's specific to our codebase.
# The user will not have any intuition about the error type, but at least they are told
# that it came from our code!
tri_solver(np.arange(9).reshape((3, 3)), 1.0)

In [ ]:
# Give a bad file path to hit an error that is covered by an in-built Python exception.
# This means that - although the error originates in our code - the user is aware that
# it is not a result of the particular details of _our code_, but a problem that can be
# encountered generally when using Python.
tri_solver(np.eye(3), 1.0, Path("/i/dont/exist/why/are/you/trying"))